# 实践 13.1  用模拟基因型比较岭回归与MLP

《统计遗传与智能育种学》第十三章 · 电子资源 `13_soydngp_virtual.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：13_soydngp_virtual.ipynb。为使核心练习不依赖图形处理器（graphics processing unit, GPU），完整可运行实践使用scikit-learn构造含加性效应和少量非线性交互效应的固定种子基因型数据。外层测试集首先锁定，开发集（development set）内部再划分训练集和验证集；Ridge与MLP在相同划分上分别使用验证集均方根误差（RMSE）选择超参数，然后用锁定设置在开发集重训，测试集只评价一次。笔记本另附SoyDNGPNext应用程序编程接口（application programming interface, API）示例。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：SEED=1301；train/validation/test=270/90/120。Ridge的validation RMSE在alpha=5/40/150时为2.727/1.981/1.694，选alpha=150；MLP三个候选配置的validation RMSE为1.900/1.841/1.840，选hidden=(48,)、alpha=0.10。锁定后Ridge的开发集Pearson相关系数r=0.896、test r=0.428、test RMSE=1.749；MLP为0.926、0.257、1.996。这个模拟中MLP并未因训练拟合更强就获得更好独立测试表现。

## 输入、完整代码、实际输出与解释

输入与可复现条件：480个模拟个体、260个标记；SEED=1301；性状由加性标记效应、一个非线性交互项和残差构成；无外部输入文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1301
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1301


In [2]:
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

n, m = 480, 260
X = rng.binomial(2, rng.uniform(0.08,0.5,m), size=(n,m)).astype(float)
b = rng.normal(0,0.12,m)
y = X@b + 0.7*(X[:,3]-1)*(X[:,17]-1) + rng.normal(0,1.3,n)
idx=np.arange(n)
dev, test = train_test_split(idx,test_size=0.25,random_state=SEED)
train, val = train_test_split(dev,test_size=0.25,random_state=SEED+1)
print('split sizes train/validation/test',len(train),len(val),len(test))


split sizes train/validation/test 270 90 120


In [3]:
def rmse(pred, obs):
    return float(np.sqrt(np.mean((np.asarray(pred)-np.asarray(obs))**2)))

def build_model(kind, param):
    if kind == 'Ridge':
        estimator = Ridge(alpha=param)
    elif kind == 'MLP':
        hidden, alpha = param
        estimator = MLPRegressor(hidden_layer_sizes=hidden, alpha=alpha,
            max_iter=500, early_stopping=True, validation_fraction=0.15,
            random_state=SEED)
    else:
        raise ValueError(kind)
    return make_pipeline(StandardScaler(), estimator)

candidates={
    'Ridge':[5,40,150],
    'MLP':[((32,),0.02),((48,),0.02),((48,),0.10)]
}
selected={}
for kind, params in candidates.items():
    scores=[]
    for param in params:
        model=build_model(kind,param)
        model.fit(X[train],y[train])
        score=rmse(model.predict(X[val]),y[val])
        scores.append(score)
        print(kind,'candidate',param,'validation RMSE',round(score,3))
    selected[kind]=params[int(np.argmin(scores))]
    print(kind,'selected',selected[kind])


Ridge candidate 5 validation RMSE 2.727
Ridge candidate 40 validation RMSE 1.981
Ridge candidate 150 validation RMSE 1.694
Ridge selected 150
MLP candidate ((32,), 0.02) validation RMSE 1.9


MLP candidate ((48,), 0.02) validation RMSE 1.841
MLP candidate ((48,), 0.1) validation RMSE 1.84
MLP selected ((48,), 0.1)


In [4]:
for kind in ['Ridge','MLP']:
    model=build_model(kind,selected[kind])
    model.fit(X[dev],y[dev])
    pred_dev=model.predict(X[dev])
    pred_test=model.predict(X[test])
    print(kind,
          'development r',round(float(np.corrcoef(pred_dev,y[dev])[0,1]),3),
          'test r',round(float(np.corrcoef(pred_test,y[test])[0,1]),3),
          'test RMSE',round(rmse(pred_test,y[test]),3))


Ridge development r 0.896 test r 0.428 test RMSE 1.749


MLP development r 0.926 test r 0.257 test RMSE 1.996


结果解释：MLP在训练数据内另作随机留出以早停，开发集的外部验证子集用于选择超参数。在当前样本量和噪声下，Ridge在外层测试集上取得更高相关和更低RMSE。该结果只说明复杂模型需要与有竞争力的统计基线在同一验证设计下比较，不能据此推断Ridge在一般数据条件下优于神经网络。

可修改练习：把样本量减半，再把非线性交互系数从0.7增到2.0；每次都重新执行内层模型选择和外层测试，不能依据外层测试结果反向选择超参数。

**SoyDNGPNext 应用程序编程接口示例（见正文13.4节；需按官方环境安装后运行，本 notebook 不执行）**

```python
# 当前官方仓库文档中的核心接口示例（运行真实SoyDNGPNext前应按官方环境准备CUDA/依赖）
from SoyDNGPNext import Train
t = Train("data/genotype.vcf", "data/traits.csv")
t.train_n(epoch=100)

import SoyDNGPNext as sn
f = sn.Forward(["MG", "protein"], 10)
f.run("data/10_test_examples.vcf")
```